# Cluster-Analyse: k-means vs. DBSCAN

**Goodie-Modul, Algorithmen & Datenstrukturen**

Ziel dieser Einheit: Wir sehen anhand von DBSCAN, wie ein eigentlich zu
langsamer Algorithmus ($\mathcal{O}(n^2)$ in der naiven Form) durch eine
geeignete räumliche Indexstruktur (k-d Tree) auf $\mathcal{O}(n \log n)$
fällt — also genau das Muster, das wir aus dem Hash-Table- und
Tree-Kapitel schon kennen: *die richtige Datenstruktur rettet den Algorithmus.*

Bitte arbeiten Sie dieses Notebook durch. 
Wodurch wird der DBSCAN massiv beschleunigt ? Wie ändert sich die algorithmische Komplexität ?

Nacher wollen wir k-means und DBSCAN hinsichtlich Eigenschaften und Performance vergleichen.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans
from sklearn.neighbors import KDTree
import time

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (10, 4)

## 1. k-means: Stärken und Grenzen

k-means findet $k$ Cluster, indem es iterativ Schwerpunkte verschiebt und
Punkte ihrem nächsten Schwerpunkt zuordnet. Komplexität pro Iteration:
$\mathcal{O}(n \cdot k \cdot d)$ — sehr schnell.

**Aber:** k-means optimiert die Summe der quadrierten Abstände zu den
Schwerpunkten. Das setzt implizit voraus, dass Cluster *konvex und
ungefähr kugelförmig* sind. An nicht-konvexen Strukturen scheitert es
systematisch.

In [ ]:
# Fall 1: drei kugelförmige Blobs - k-means ist hier ideal
X_blobs, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42)

# Fall 2: zwei verschränkte Halbmonde - geometrisch zusammenhängend, aber nicht konvex
X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

km1 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_blobs)
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=km1.labels_, cmap="tab10", s=15)
axes[0].set_title("k-means auf Blobs: alles gut")

km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=km2.labels_, cmap="tab10", s=15)
axes[1].set_title("k-means auf Moons: schneidet die Monde durch")

plt.tight_layout(); plt.show()

Die rechte Seite zeigt das Problem deutlich: k-means trennt die beiden
Halbmonde mit einer geraden Linie durch die Mitte, statt der tatsächlichen
geometrischen Struktur zu folgen. Wir brauchen einen Algorithmus, der
*Dichte* statt Abstand zu einem Schwerpunkt betrachtet.

## 2. DBSCAN: Idee

**Density-Based Spatial Clustering of Applications with Noise** (Ester et al., 1996).

Zwei Parameter:
- $\varepsilon$: Radius der Nachbarschaft
- $\text{minPts}$: Mindestanzahl Punkte in der $\varepsilon$-Nachbarschaft

Punkt-Klassifikation:
- **Kernpunkt:** mindestens $\text{minPts}$ Punkte in seiner $\varepsilon$-Nachbarschaft
- **Randpunkt:** kein Kernpunkt, aber in der Nachbarschaft eines Kernpunktes
- **Rauschen:** weder Kern- noch Randpunkt

Cluster = transitive Hülle von Kernpunkten unter Erreichbarkeit + ihre Randpunkte.

**Komplexität:** Der Algorithmus selbst hat einen Aussen-Loop über alle $n$
Punkte. Pro Punkt führt er einen *Range Query* durch: alle Nachbarn im
Radius $\varepsilon$ finden. Damit ist die Gesamtkomplexität
$\mathcal{O}(n \cdot Q(n))$, wobei $Q(n)$ die Kosten eines Range Query sind.

## 3. Naive Implementierung — $\mathcal{O}(n^2)$

Range Query hier brute force: für jeden Punkt alle anderen Punkte
durchgehen und Distanz prüfen. Das ist $\mathcal{O}(n)$ pro Anfrage,
$\mathcal{O}(n^2)$ insgesamt.

In [ ]:
def dbscan_naive(X, eps, min_pts):
    """DBSCAN mit linearer Range-Query. Komplexitaet: O(n^2)."""
    n = len(X)
    labels = np.full(n, -1)        # -1 = noch nicht zugewiesen / Rauschen
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    def region_query(p):
        # O(n) - der Flaschenhals
        return [j for j in range(n) if np.linalg.norm(X[p] - X[j]) <= eps]

    for p in range(n):
        if visited[p]:
            continue
        visited[p] = True
        neighbors = region_query(p)
        if len(neighbors) < min_pts:
            labels[p] = -1            # vorerst Rauschen (kann Randpunkt werden)
            continue
        # Neuen Cluster aufbauen
        labels[p] = cluster_id
        seeds = list(neighbors)
        i = 0
        while i < len(seeds):
            q = seeds[i]
            if not visited[q]:
                visited[q] = True
                q_neighbors = region_query(q)
                if len(q_neighbors) >= min_pts:
                    seeds.extend(q_neighbors)   # Cluster waechst weiter
            if labels[q] == -1:
                labels[q] = cluster_id          # Randpunkt zuweisen
            i += 1
        cluster_id += 1

    return labels

In [ ]:
labels_naive = dbscan_naive(X_moons, eps=0.2, min_pts=5)

plt.figure(figsize=(6, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels_naive, cmap="tab10", s=15)
plt.title(f"DBSCAN (naiv) auf Moons: {len(set(labels_naive)) - (1 if -1 in labels_naive else 0)} Cluster, "
          f"{(labels_naive == -1).sum()} Rauschpunkte")
plt.show()

DBSCAN trennt die Halbmonde sauber, weil es der Dichte folgt — keine
implizite Konvexitätsannahme.

## 4. DBSCAN mit k-d Tree — $\mathcal{O}(n \log n)$

Der Algorithmus selbst ändert sich nicht. Wir tauschen nur die
Range-Query-Implementierung aus. Der k-d Tree wird einmal in
$\mathcal{O}(n \log n)$ aufgebaut; ein Range Query kostet im Mittel
$\mathcal{O}(\log n)$ in niedrigen Dimensionen.

In [ ]:
def dbscan_kdtree(X, eps, min_pts):
    """DBSCAN mit k-d Tree als Range-Query-Index. Komplexitaet: O(n log n)."""
    n = len(X)
    tree = KDTree(X)              # einmaliger Aufbau: O(n log n)
    labels = np.full(n, -1)
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    def region_query(p):
        # O(log n) im Mittel statt O(n)
        return tree.query_radius(X[p:p+1], r=eps)[0].tolist()

    # Ab hier: identische Logik wie dbscan_naive
    for p in range(n):
        if visited[p]:
            continue
        visited[p] = True
        neighbors = region_query(p)
        if len(neighbors) < min_pts:
            labels[p] = -1
            continue
        labels[p] = cluster_id
        seeds = list(neighbors)
        i = 0
        while i < len(seeds):
            q = seeds[i]
            if not visited[q]:
                visited[q] = True
                q_neighbors = region_query(q)
                if len(q_neighbors) >= min_pts:
                    seeds.extend(q_neighbors)
            if labels[q] == -1:
                labels[q] = cluster_id
            i += 1
        cluster_id += 1

    return labels

In [ ]:
# Sanity Check: gleiches Ergebnis wie naive Variante?
labels_kd = dbscan_kdtree(X_moons, eps=0.2, min_pts=5)

# Cluster-Labels sind nicht zwingend identisch nummeriert; vergleiche stattdessen die Partition
def same_partition(a, b):
    """Vergleicht zwei Label-Vektoren bis auf Permutation der Cluster-IDs."""
    from collections import defaultdict
    mapping = {}
    for x, y in zip(a, b):
        if x in mapping:
            if mapping[x] != y: return False
        else:
            mapping[x] = y
    return True

print("Naiv und KDTree liefern dieselbe Partition:", same_partition(labels_naive, labels_kd))

## 5. Laufzeitvergleich

Wir messen beide Varianten für wachsendes $n$ bei festen
$\varepsilon, \text{minPts}$. Erwartung: die naive Variante zeigt
ein klar quadratisches Wachstum, die KDTree-Variante liegt deutlich
darunter.

In [ ]:
sizes = [200, 500, 1000, 2000, 4000]
times_naive = []
times_kd = []

for n in sizes:
    X, _ = make_moons(n_samples=n, noise=0.05, random_state=42)

    t0 = time.perf_counter()
    dbscan_naive(X, eps=0.2, min_pts=5)
    times_naive.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    dbscan_kdtree(X, eps=0.2, min_pts=5)
    times_kd.append(time.perf_counter() - t0)

    print(f"n={n:5d}   naiv: {times_naive[-1]:7.3f}s   kdtree: {times_kd[-1]:7.3f}s   "
          f"Faktor: {times_naive[-1]/times_kd[-1]:5.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sizes, times_naive, "o-", label="naiv  O(n^2)")
axes[0].plot(sizes, times_kd,    "s-", label="kdtree O(n log n)")
axes[0].set_xlabel("n"); axes[0].set_ylabel("Zeit [s]")
axes[0].set_title("Linear")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].loglog(sizes, times_naive, "o-", label="naiv  O(n^2)")
axes[1].loglog(sizes, times_kd,    "s-", label="kdtree O(n log n)")
# Referenzgeraden: Steigung 2 vs. Steigung ~1
ref_n2 = np.array(sizes, dtype=float); ref_n2 = ref_n2**2 * times_naive[0] / sizes[0]**2
ref_nl = np.array(sizes, dtype=float); ref_nl = ref_nl * np.log(ref_nl) * times_kd[0] / (sizes[0] * np.log(sizes[0]))
axes[1].loglog(sizes, ref_n2, "k--", alpha=0.4, label="Referenz n^2")
axes[1].loglog(sizes, ref_nl, "k:",  alpha=0.4, label="Referenz n log n")
axes[1].set_xlabel("n"); axes[1].set_ylabel("Zeit [s]")
axes[1].set_title("log-log")
axes[1].legend(); axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout(); plt.show()

Im log-log-Plot lesen wir die Steigung als Exponenten ab: Welche Steigung haben die beiden Kurven (=Geraden im log log) ?